Path

In [ ]:
import os
import geopandas as gpd

ROOT = r"__Data/Data Source 2025" # Row EVCS data
RESULT = r"__Data/Global_Country_2025" # Path where the merged EVCS data is saved

# Other source data
BUILT_UP = r"_Data/built_up_area/builtUpArea_fixGeom.gpkg" # Built-up area data
POPS = r"__Data/population_All" # Population data by country
POP = r"__Data/Population_Related/global_pop_2025_CN_1km_R2025A_UA_v1.tif" # Global population data
ROADS = r"__Data/roadsGraph_Clean" # Road network data
POI_ROW = r"__Data/extract.parquet" # Row vector POI data
POI = r"__Data/globalPOI_2025" # Processed raster POI data
GDP = r"__Data/global_gdp/gdp_2020.tif"

# Fix save path, do not modify
ROADS_RESULT = os.path.join(RESULT, "density_roads") # Path where the calculated road density data is saved

Merge row data

In [ ]:
from function import mergeByMaxRecord, merge
merge(ROOT, RESULT, 0)
mergeByMaxRecord(RESULT)

Add population and built-up area data into road graph data

In [ ]:
from analysis import calPop
# Country list
countries = {
    "CAN", "USA", "CHN", "ZAF", "CYP", "IRL", "GRC", "ISL", "ESP", "MLT",
    "ITA", "PRT", "DEU", "CHE", "BGR", "FRA", "HRV", "ROU", "UKR", "SVN",
    "HUN", "MDA", "SWE", "AUT", "SVK", "CZE", "POL", "LUX", "BEL", "NLD",
    "DNK", "LVA", "EST", "NOR", "FIN", "LTU", "JPN", "HKG", "KOR", "MAC",
    "NZL", "AND", "GBR", "DOM", "EGY", "GIB", "IMN", "MCO", "NAM", "PSE",
    "SMR", "SWZ", "UGA", "XKX", "AUS", "JOR", "SGP", "URY", "MYS", "MEX",
    "GEO", "ALB", "SRB", "TWN", "RUS", "ARE", "ABW", "BRA", "IND", "TUR",
    "CRI", "BLR", "MAR", "PRY", "THA", "KAZ", "ISR", "CHL", "AZE", "BIH",
    "BHS", "BRB", "PRI", "ARG", "GGY", "ARM", "SAU", "MUS", "TUN", "KEN",
    "JAM", "GLP", "COL", "JEY", "GTM", "PAN", "REU", "CUW", "TTO", "MKD",
    "IDN", "NPL", "UZB", "LIE", "PHL", "LKA", "VNM", "RWA", "ECU", "MNE",
    "TJK", "BOL", "QAT", "BRN", "ALA", "HND", "KHM", "IRQ", "KWT", "BHR",
    "SLV", "TCA", "OMN", "BWA", "VIR", "LAO", "KGZ", "CYM", "MTQ", "PER",
    "DMA", "ETH", "GUM", "LSO", "VCT", "FRO", "GRL", "MNG", "NCL", "PYF",
    "GRD", "BLZ", "ZWE", "PAK", "GHA", "FJI", "BMU", "NIC", "ZMB", "VEN",
    "LBN", "MDG", "DZA", "LCA", "SJM", "MMR", "NFK", "HTI", "ATG", "CPV",
    "MLI", "ESH", "CIV", "BTN", "TZA", "SXM", "MNP", "SHN", "MAF", "NGA",
    "MRT", "BGD", "GUF", "AFG", "SYR", "EUR_Merge"
}

# Built-up area data with ISO3 code
urban = gpd.read_file(BUILT_UP, encoding="utf-8")

calPop(countries, (urban, "iso3"), POPS, ROADS_RESULT, 32)

Calculate road density

In [ ]:
from analysis import calDensity, generate, urbanConnecting
for buffer in [1000, 3000, 5000]:
    a = calDensity(RESULT, maxThread=64, buffer=buffer)
    a.calDensity(
        ROADS, # Root path of roads data
        ("mainRoad_{}".format(buffer), {"motorway", "motorway_link", "trunk", "trunk_link", "primary", "primary_link"}),
        ROADS_RESULT
    )
    a.saveResult()

# Calculate the OD pair
urbanConnecting(RESULT, maxThread=64)

# Merge the results into a single file
generate(ROADS_RESULT)

Convert POI vector data into raster with the same shape of GDP raster

In [ ]:
from analysis import points2Raster
a = points2Raster(GDP, POI_ROW)
a.process(POI, 1) # Admin and public
a.process(POI, 2) # Commercial and entertainment
a.process(POI, 3) # Leasure and tourism
a.process(POI) # All the above three categories

Cal POI, Population, and GDP density (by bulit-up area)

In [ ]:
a = calDensity(RESULT, roadsPath=ROADS_RESULT, maxThread=32)
a.calDensity(os.path.join(POI, "POI_1.tif"), "POI_01")
a.calDensity(os.path.join(POI, "POI_2.tif"), "POI_02")
a.calDensity(os.path.join(POI, "POI_3.tif"), "POI_03")
a.calDensity(os.path.join(POI, "POI_ALL.tif"), "POI_ALL")
a.calDensity(POP, "POP")
a.calDensity(GDP, "GDP")
a.saveResult()